# From-Scratch vs. Established Libraries

Sanity-checks the from-scratch implementations in `metaheuristics.algorithms` against [mealpy](https://mealpy.readthedocs.io/) (covers all five algorithms under one API) and `scipy.optimize` (`differential_evolution`, `dual_annealing`) as a second, ubiquitous baseline. This is **illustrative, not a rigorous benchmark** — budgets are matched loosely (population size × generations/iterations), not tuned per algorithm.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from mealpy import DE as mDE
from mealpy import GA as mGA
from mealpy import PSO as mPSO
from mealpy import SA as mSA
from mealpy import SMA as mSMA
from mealpy import FloatVar
from scipy.optimize import differential_evolution, dual_annealing

from metaheuristics.algorithms.differential_evolution import DifferentialEvolution
from metaheuristics.algorithms.genetic_algorithm import GeneticAlgorithm
from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization
from metaheuristics.algorithms.simulated_annealing import SimulatedAnnealing
from metaheuristics.algorithms.slime_mould import SlimeMouldAlgorithm
from metaheuristics.benchmarks import landscapes as L

POP, GEN = 40, 80
FUNCTIONS = [L.rastrigin, L.ackley]

In [2]:
def mealpy_problem(func):
    lb = [b[0] for b in func.bounds]
    ub = [b[1] for b in func.bounds]
    return {
        'obj_func': lambda x: float(func(np.asarray(x))),
        'bounds': FloatVar(lb=lb, ub=ub),
        'minmax': 'min',
        'log_to': None,
    }

rows, curves = [], {}

for func in FUNCTIONS:
    ours = {
        'GA (ours)': GeneticAlgorithm(population_size=POP, max_generations=GEN),
        'PSO (ours)': ParticleSwarmOptimization(num_particles=POP, max_iterations=GEN),
        'SA (ours)': SimulatedAnnealing(max_iterations=POP * GEN),
        'DE (ours)': DifferentialEvolution(population_size=POP, max_generations=GEN),
        'SMO (ours)': SlimeMouldAlgorithm(population_size=POP, max_iterations=GEN),
    }
    for name, algo in ours.items():
        np.random.seed(0)
        result = algo.optimize(func, func.bounds)
        rows.append({'function': func.__name__, 'algorithm': name, 'best_fitness': result.best_fitness})
        curves[(func.__name__, name)] = result.fitness_history

    problem = mealpy_problem(func)
    mealpy_algos = {
        'GA (mealpy)': mGA.BaseGA(epoch=GEN, pop_size=POP),
        'PSO (mealpy)': mPSO.OriginalPSO(epoch=GEN, pop_size=POP),
        'SA (mealpy)': mSA.OriginalSA(epoch=GEN, pop_size=POP),
        'DE (mealpy)': mDE.OriginalDE(epoch=GEN, pop_size=POP),
        'SMO (mealpy)': mSMA.OriginalSMA(epoch=GEN, pop_size=POP),
    }
    for name, model in mealpy_algos.items():
        g_best = model.solve(problem, seed=0)
        rows.append({'function': func.__name__, 'algorithm': name, 'best_fitness': g_best.target.fitness})
        curves[(func.__name__, name)] = model.history.list_global_best_fit

    de = differential_evolution(lambda x, func=func: func(np.asarray(x)), bounds=func.bounds, maxiter=GEN, popsize=15, seed=0)
    rows.append({'function': func.__name__, 'algorithm': 'DE (scipy)', 'best_fitness': de.fun})

    sa = dual_annealing(lambda x, func=func: func(np.asarray(x)), bounds=func.bounds, maxiter=1000, seed=0)
    rows.append({'function': func.__name__, 'algorithm': 'SA (scipy)', 'best_fitness': sa.fun})

df = pd.DataFrame(rows)
df.pivot(index='algorithm', columns='function', values='best_fitness').sort_index()

function,ackley,rastrigin
algorithm,,
DE (mealpy),8.339012e-06,9.949591e-01
DE (ours),9.821790e-09,1.351759e-06
DE (scipy),1.598721e-13,0.000000e+00
GA (mealpy),9.057654e-02,1.380944e-01
GA (ours),3.054904e-04,2.421852e-06
PSO (mealpy),1.421085e-14,9.949591e-01
PSO (ours),4.332595e-06,3.804992e-10
SA (mealpy),2.267372e-01,5.012580e+00
SA (ours),7.909097e-02,4.700437e-01


## Convergence curves on Rastrigin: ours vs. mealpy

In [3]:
fig = go.Figure()
for name in ['GA (ours)', 'PSO (ours)', 'SA (ours)', 'DE (ours)', 'SMO (ours)',
             'GA (mealpy)', 'PSO (mealpy)', 'SA (mealpy)', 'DE (mealpy)', 'SMO (mealpy)']:
    fig.add_trace(go.Scatter(y=curves[('rastrigin', name)], name=name))
fig.update_layout(
    title='Convergence on Rastrigin', xaxis_title='iteration', yaxis_title='best fitness',
    yaxis_type='log',
)
fig.show()